# Fundamentos de Bases de Datos — Pipeline SQLite (E-commerce)

Inspirado en la ruta de [VibePixel Academy — Baby Steps Tech](https://haroldsthid.github.io/VibePixel-AcademyBabyStepsTech/index.html#ruta).

Este notebook recorre un pipeline completo de fundamentos de bases de datos usando **SQLite** como motor:

1. **Diseño del Modelo ERM** — entidades, atributos y relaciones de un dominio de e-commerce.
2. **Poblamiento de tablas** — datos sintéticos generados con `Faker`.
3. **Data Cleaning** — se inyectan problemas de calidad de datos típicos (duplicados, nulos, formatos inconsistentes) y se limpian con `pandas`.
4. **Carga en SQLite** — persistencia del modelo limpio en una base `ecommerce.db`.
5. **Reporting** — queries SQL que sirven de insumo para un proceso de reporting, con visualización en `matplotlib`.

> Abrí este notebook en Google Colab (botón "Open in Colab" en el README del repo) para ejecutarlo de punta a punta sin instalar nada localmente.


## 1. Setup y dependencias

Instalamos `Faker` (no viene por defecto en Colab) y fijamos una semilla aleatoria para que el dataset generado sea reproducible.

In [ ]:
!pip install -q faker


In [ ]:
import sqlite3
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from faker import Faker

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker("es_AR")
fake.seed_instance(SEED)

DB_PATH = "ecommerce.db"


## 2. Diseño del Modelo ERM

Dominio: **e-commerce**. Seis entidades cubren el flujo completo desde el cliente hasta el pago de una orden.

```
customers (1) ───< (N) orders (1) ───< (N) order_items >─── (1) products (N) >─── (1) categories
                        │
                        └───< (1) payments
```

### Entidades y atributos

| Entidad | PK | Atributos | FKs |
|---|---|---|---|
| `customers` | `customer_id` | first_name, last_name, email, phone, city, country, signup_date | — |
| `categories` | `category_id` | category_name | — |
| `products` | `product_id` | product_name, price, stock_quantity | `category_id` → categories |
| `orders` | `order_id` | order_date, status | `customer_id` → customers |
| `order_items` | `order_item_id` | quantity, unit_price | `order_id` → orders, `product_id` → products |
| `payments` | `payment_id` | payment_date, amount, payment_method | `order_id` → orders |

### Reglas de negocio (cardinalidad)
- Un **cliente** puede tener **0..N** órdenes; toda orden pertenece a **exactamente 1** cliente.
- Una **orden** tiene **1..N** líneas (`order_items`) y **exactamente 1** pago (`payments`).
- Un **producto** pertenece a **exactamente 1** categoría; una categoría agrupa **0..N** productos.
- `order_items` es la tabla asociativa que resuelve la relación **N a N** entre `orders` y `products`.

### DDL (script de creación de esquema)


In [ ]:
SCHEMA_SQL = '''
CREATE TABLE customers (
    customer_id   INTEGER PRIMARY KEY,
    first_name    TEXT NOT NULL,
    last_name     TEXT NOT NULL,
    email         TEXT NOT NULL UNIQUE,
    phone         TEXT,
    city          TEXT,
    country       TEXT NOT NULL,
    signup_date   TEXT NOT NULL
);

CREATE TABLE categories (
    category_id   INTEGER PRIMARY KEY,
    category_name TEXT NOT NULL UNIQUE
);

CREATE TABLE products (
    product_id    INTEGER PRIMARY KEY,
    product_name  TEXT NOT NULL,
    category_id   INTEGER NOT NULL REFERENCES categories(category_id),
    price          REAL NOT NULL CHECK (price >= 0),
    stock_quantity INTEGER NOT NULL CHECK (stock_quantity >= 0)
);

CREATE TABLE orders (
    order_id      INTEGER PRIMARY KEY,
    customer_id   INTEGER NOT NULL REFERENCES customers(customer_id),
    order_date    TEXT NOT NULL,
    status        TEXT NOT NULL CHECK (status IN ('pending', 'paid', 'shipped', 'cancelled'))
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id      INTEGER NOT NULL REFERENCES orders(order_id),
    product_id    INTEGER NOT NULL REFERENCES products(product_id),
    quantity      INTEGER NOT NULL CHECK (quantity > 0),
    unit_price    REAL NOT NULL CHECK (unit_price >= 0)
);

CREATE TABLE payments (
    payment_id     INTEGER PRIMARY KEY,
    order_id       INTEGER NOT NULL UNIQUE REFERENCES orders(order_id),
    payment_date   TEXT NOT NULL,
    amount         REAL NOT NULL CHECK (amount >= 0),
    payment_method TEXT NOT NULL CHECK (payment_method IN ('credit_card', 'debit_card', 'bank_transfer', 'cash'))
);
'''


## 3. Poblamiento de tablas con Faker

Generamos datos sintéticos para cada entidad. A propósito, inyectamos problemas de calidad típicos de un dataset "crudo" (duplicados, nulos, formatos inconsistentes) para poder practicar el data cleaning en la sección siguiente.


In [ ]:
N_CUSTOMERS = 200
N_CATEGORIES = 8
N_PRODUCTS = 60
N_ORDERS = 500

CATEGORY_NAMES = [
    "Electrónica", "Hogar", "Indumentaria", "Deportes",
    "Libros", "Juguetes", "Belleza", "Alimentos",
]

COUNTRY_VARIANTS = ["Argentina", "argentina", "ARGENTINA", "Chile", "chile", "Uruguay", " Uruguay "]


In [ ]:
# --- customers (con duplicados, emails nulos y teléfonos con formato inconsistente) ---
customers_raw = []
for i in range(1, N_CUSTOMERS + 1):
    customers_raw.append({
        "customer_id": i,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.email() if random.random() > 0.05 else None,
        "phone": random.choice([
            fake.phone_number(),
            fake.phone_number().replace("-", " "),
            fake.msisdn(),
        ]),
        "city": fake.city(),
        "country": random.choice(COUNTRY_VARIANTS),
        "signup_date": fake.date_between(start_date="-3y", end_date="today").isoformat(),
    })

# duplicamos ~4% de los clientes (mismo customer_id reasignado con nueva PK, pero mismo email)
for _ in range(8):
    dup = random.choice(customers_raw).copy()
    dup["customer_id"] = len(customers_raw) + 1
    customers_raw.append(dup)

customers_df = pd.DataFrame(customers_raw)
customers_df.head()


In [ ]:
# --- categories ---
categories_df = pd.DataFrame({
    "category_id": range(1, N_CATEGORIES + 1),
    "category_name": CATEGORY_NAMES[:N_CATEGORIES],
})
categories_df


In [ ]:
# --- products (precio como texto con símbolo de moneda, para forzar limpieza de tipos) ---
products_raw = []
for i in range(1, N_PRODUCTS + 1):
    price = round(random.uniform(500, 250000), 2)
    products_raw.append({
        "product_id": i,
        "product_name": fake.catch_phrase(),
        "category_id": random.randint(1, N_CATEGORIES),
        "price": f"$ {price:,.2f}",
        "stock_quantity": random.randint(0, 500),
    })

products_df = pd.DataFrame(products_raw)
products_df.head()


In [ ]:
# --- orders ---
STATUS_CHOICES = ["pending", "paid", "shipped", "cancelled"]
valid_customer_ids = customers_df["customer_id"].tolist()

orders_raw = []
for i in range(1, N_ORDERS + 1):
    orders_raw.append({
        "order_id": i,
        "customer_id": random.choice(valid_customer_ids),
        "order_date": fake.date_time_between(start_date="-2y", end_date="now").isoformat(sep=" "),
        "status": random.choice(STATUS_CHOICES),
    })

orders_df = pd.DataFrame(orders_raw)
orders_df.head()


In [ ]:
# --- order_items (1 a 4 líneas por orden) ---
order_items_raw = []
item_id = 1
for order_id in orders_df["order_id"]:
    for _ in range(random.randint(1, 4)):
        product = products_df.sample(1).iloc[0]
        unit_price = float(str(product["price"]).replace("$", "").replace(",", "").strip())
        order_items_raw.append({
            "order_item_id": item_id,
            "order_id": order_id,
            "product_id": product["product_id"],
            "quantity": random.randint(1, 5),
            "unit_price": unit_price,
        })
        item_id += 1

order_items_df = pd.DataFrame(order_items_raw)
order_items_df.head()


In [ ]:
# --- payments (algunas órdenes sin pago todavía, simula 'pending') ---
PAYMENT_METHODS = ["credit_card", "debit_card", "bank_transfer", "cash"]

payments_raw = []
payment_id = 1
for _, order in orders_df.iterrows():
    if order["status"] == "pending":
        continue  # aún no tiene pago asociado
    order_total = order_items_df.loc[order_items_df["order_id"] == order["order_id"],
                                      ["quantity", "unit_price"]].prod(axis=1).sum()
    payments_raw.append({
        "payment_id": payment_id,
        "order_id": order["order_id"],
        "payment_date": order["order_date"],
        "amount": round(order_total, 2),
        "payment_method": random.choice(PAYMENT_METHODS),
    })
    payment_id += 1

payments_df = pd.DataFrame(payments_raw)
payments_df.head()


## 4. Data Cleaning

Los datos generados en la sección anterior traen a propósito los problemas más comunes de un dataset crudo. Los resolvemos uno por uno con `pandas`:

- **Duplicados** en `customers` (mismo email, distinto `customer_id`).
- **Emails nulos** → se eliminan las filas (no hay forma de recuperar el dato).
- **Teléfonos con formato inconsistente** → se normalizan a solo dígitos.
- **País con casing/espacios inconsistentes** → se normaliza a *Title Case* y se recorta.
- **Precio como texto con símbolo de moneda** → se convierte a `float`.


In [ ]:
# 4.1 — Duplicados por email en customers
before = len(customers_df)
customers_clean = customers_df.drop_duplicates(subset="email", keep="first").copy()
print(f"customers: {before} -> {len(customers_clean)} filas (se eliminaron {before - len(customers_clean)} duplicados)")


In [ ]:
# 4.2 — Emails nulos: no se pueden imputar, se descartan
before = len(customers_clean)
customers_clean = customers_clean.dropna(subset=["email"]).copy()
print(f"customers: {before} -> {len(customers_clean)} filas (se eliminaron {before - len(customers_clean)} sin email)")


In [ ]:
# 4.3 — Normalización de teléfono (solo dígitos) y país (Title Case, sin espacios)
customers_clean["phone"] = customers_clean["phone"].str.replace(r"\D", "", regex=True)
customers_clean["country"] = customers_clean["country"].str.strip().str.title()

customers_clean[["phone", "country"]].head()


In [ ]:
# 4.4 — Precio de productos: de texto '$ 12,345.67' a float
products_clean = products_df.copy()
products_clean["price"] = (
    products_clean["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .astype(float)
)
products_clean.head()


In [ ]:
# 4.5 — Integridad referencial: descartamos orders/order_items que quedaron huérfanos
# tras eliminar clientes duplicados/sin email en el paso 4.1-4.2
valid_customer_ids = set(customers_clean["customer_id"])
orders_clean = orders_df[orders_df["customer_id"].isin(valid_customer_ids)].copy()

valid_order_ids = set(orders_clean["order_id"])
order_items_clean = order_items_df[order_items_df["order_id"].isin(valid_order_ids)].copy()
payments_clean = payments_df[payments_df["order_id"].isin(valid_order_ids)].copy()

categories_clean = categories_df.copy()

print(f"orders: {len(orders_df)} -> {len(orders_clean)}")
print(f"order_items: {len(order_items_df)} -> {len(order_items_clean)}")
print(f"payments: {len(payments_df)} -> {len(payments_clean)}")


## 5. Carga en SQLite

Creamos el esquema definido en la sección 2 y cargamos los DataFrames ya limpios en `ecommerce.db`.


In [ ]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.executescript("DROP TABLE IF EXISTS payments; DROP TABLE IF EXISTS order_items; "
                     "DROP TABLE IF EXISTS orders; DROP TABLE IF EXISTS products; "
                     "DROP TABLE IF EXISTS categories; DROP TABLE IF EXISTS customers;")
cursor.executescript(SCHEMA_SQL)
conn.commit()
print("Esquema creado en", DB_PATH)


In [ ]:
tables = {
    "customers": customers_clean,
    "categories": categories_clean,
    "products": products_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "payments": payments_clean,
}

for name, df in tables.items():
    df.to_sql(name, conn, if_exists="append", index=False)

conn.commit()

for name in tables:
    count = cursor.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"{name}: {count} filas cargadas")


## 6. Queries de Reporting

Tres consultas SQL pensadas como insumo directo para un dashboard de reporting: ingresos por categoría y mes, top clientes por gasto, y ticket promedio por país.


### 6.1 — Ingresos por categoría y mes

In [ ]:
query_revenue_by_category = '''
SELECT
    c.category_name,
    strftime('%Y-%m', o.order_date) AS month,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
JOIN categories c ON c.category_id = p.category_id
WHERE o.status != 'cancelled'
GROUP BY c.category_name, month
ORDER BY month, revenue DESC;
'''

revenue_by_category = pd.read_sql(query_revenue_by_category, conn)
revenue_by_category.head(10)


In [ ]:
pivot = revenue_by_category.pivot(index="month", columns="category_name", values="revenue").fillna(0)
pivot.plot(kind="bar", stacked=True, figsize=(12, 6), colormap="tab20")
plt.title("Ingresos por categoría y mes")
plt.ylabel("Ingresos ($)")
plt.xlabel("Mes")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


### 6.2 — Top 10 clientes por gasto total

In [ ]:
query_top_customers = '''
SELECT
    cu.customer_id,
    cu.first_name || ' ' || cu.last_name AS customer_name,
    ROUND(SUM(p.amount), 2) AS total_spent
FROM payments p
JOIN orders o ON o.order_id = p.order_id
JOIN customers cu ON cu.customer_id = o.customer_id
GROUP BY cu.customer_id
ORDER BY total_spent DESC
LIMIT 10;
'''

top_customers = pd.read_sql(query_top_customers, conn)
top_customers


In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top_customers["customer_name"], top_customers["total_spent"], color="#4C72B0")
plt.gca().invert_yaxis()
plt.title("Top 10 clientes por gasto total")
plt.xlabel("Gasto total ($)")
plt.tight_layout()
plt.show()


### 6.3 — Ticket promedio por país

In [ ]:
query_avg_order_by_country = '''
SELECT
    cu.country,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(AVG(p.amount), 2) AS avg_order_value
FROM payments p
JOIN orders o ON o.order_id = p.order_id
JOIN customers cu ON cu.customer_id = o.customer_id
GROUP BY cu.country
ORDER BY avg_order_value DESC;
'''

avg_order_by_country = pd.read_sql(query_avg_order_by_country, conn)
avg_order_by_country


In [ ]:
conn.close()
print(f"Base de datos disponible en: {DB_PATH}")
print("En Colab: descargala desde el panel de archivos (icono de carpeta a la izquierda).")


## Próximos pasos

- Agregar más queries de reporting (ej. cohort de retención de clientes, productos con stock crítico).
- Exportar `ecommerce.db` y conectarlo desde una herramienta de BI (Metabase, Looker Studio vía conector SQLite/CSV).
- Versionar el esquema (`SCHEMA_SQL`) como migraciones si el modelo evoluciona.
